# 3D multichannel segmentation

This notebook demonstrates a complete workflow for segmenting nuclei, cytoplasm, and intracellular structures from **3D microscopy data**, and for quantifying structures at both the object and cell level. The pipeline is modular and extensible, allowing different segmentation strategies depending on the channel and biological question.

The general pipeline includes:

1. Loading and preprocessing volumetric data

    - Intensity normalization
    - Optional downsampling and smoothing
2. Segmentation of cellular compartments

    - Nuclei (using deep learning models: StarDist or Cellpose)

    - Cytoplasm (via intensity- or membrane-based watershed)

    - Detection of intracellular structures

        -AICS segmentation workflows (e.g., dot/filament filters)
3. (optional) Post-processing (object removal, smoothing)

4. Quantification of structures per cell

    - Mapping detected objects to their parent cell

    - Extracting per-object and per-cell features (count, volume, intensity)

    - Exporting results to CSV for downstream analysis

5. Quality control

    - Visual overlays (e.g., maximum intensity projections, per-cell structure maps)
    
The goal is to link each detected intracellular structure to its cell of origin, enabling biologically meaningful, cell-level measurements.

## Load a 3D image 

You need to specify your input folder, how many images to segment, if you want to dowsample, normalise per slice, and the map your channel into a directory:

    - "nucleus": 0 → channel index 0 contains the nuclear stain.

    - "cytoplasm": 2 → channel index 2 contains cytoplasmic signal.

    - "intracellular": 1 → channel index 1 contains the organelle/structure of interest.
    
This mapping depends on the acquisition setup and might change between datasets — check the raw image metadata if in doubt (e.g. via opening a representative example image in FIJI)

In [ ]:
## Libraries
## Load packages


import skimage
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import pandas as pd
import os
import tifffile

# --------------------
# INPUT SETTINGS
# --------------------

input_folder = Path("/home/ucbtvsi/Image-Analysis-Summer-Project/raw/images")
output_folder = Path("/home/ucbtvsi/Image-Analysis-Summer-Project/output")
output_folder.mkdir(parents=True, exist_ok=True)

num_images_to_segment = 1 # add more if you think your computer can handle 

# --------------------
# --------------------

# 1. load images as multichannel dictionary

# enter voxel size of your images (obtained from the metadata)
voxel_size_um = [1, 0.48, 0.48]  # Z, Y, X; from metadata

# which channel corresponds to which structure — check your metadata if unsure
channel_map = {
    "nucleus": 0,
    "cytoplasm": 2,
    "intracellular": 1
}

# load images
files = sorted(input_folder.glob("*.tif"))[:num_images_to_segment]

all_volumes = []

for f in files:
    img = tifffile.imread(f)  # shape: (Z, C, Y, X)
    
    # extract each channel by name
    channels = {name: img[:, idx, :, :] for name, idx in channel_map.items()}
    
    all_volumes.append({
        "filename": f.stem,
        "channels": channels
    })

# preview
print(all_volumes[0]["filename"])
print(all_volumes[0]["channels"].keys())
print(all_volumes[0]["channels"]["nucleus"].shape)  # (Z, Y, X)

## Quick check: middle slice for each channel

We can display the **middle Z-slice** of each channel from the same (first) image of the folder. You can view different images by editing Volume_number.

This allows you to quickly confirm:

    - Channel order (e.g., nucleus, cytoplasm, intracellular)
    - The objects in the different channels should overlap, with nucleus and intracellular objects within the cytoplasm area.
    - That the data is loaded correctly and matches expectations
    
If the signal looks wrong (e.g., nucleus is empty but cytoplasm is bright), check the channel_map definition above.

In [ ]:
### Which volume to view?
Volume_number = 0

# take the first loaded volume
vol = all_volumes[Volume_number]["channels"]

# find middle slice
z_mid = next(iter(vol.values())).shape[0] // 2  

# plot all channels in a grid
n_channels = len(vol)
fig, axes = plt.subplots(1, n_channels, figsize=(5 * n_channels, 5))

for ax, (name, data) in zip(axes, vol.items()):
    ax.imshow(data[z_mid], cmap="gray")
    ax.set_title(f"{name} (z={z_mid})")
    ax.axis("off")

plt.tight_layout()
plt.show()

### Per-channel preprocessing, segmentation, and quantification

 Each channel is preprocessed and segmented in a separate sub-section, allowing to save and visualise intermediate steps for quality checks.

We need to define:

1. Number of images to test (num_test_volumes): Defines how many test images from the input_folder will be used for the trial analysis throughout
2. Downsampling (downsize_factor): Reduces memory usage and speeds up computation.
3. Preprocessing: Gaussian filtering and normalization (either per-slice or across the full 3D stack).

In [ ]:
# --------------------
# SETTINGS
# --------------------
demo = 0              # change to select a different image
downsize_factor = 1   # 1 = no downsampling; 0.5 to speed things up
sigma_um = 1          # gaussian smoothing in µm

# convert sigma from µm to voxels (accounts for non-isotropic voxel size)
sigma_vox = tuple(sigma_um / v for v in voxel_size_um)  # (1.0, 2.08, 2.08)

#for cellpose segmentation

gpu = True            # change to False if no gpu (will be slower)


a) **Nucleus**

This section describes the preprocessing, segmentation, and quantification of nuclei in 3D microscopy images. The workflow uses optionally either StarDist or Cellpose models for segmentation, with post-processing and quality control steps available.

1. **Preprocessing**: normalize nucleus channel, apply Gaussian filter to reduce noise, optionally downsample.

2. **Segmentation**: using DL model:

    - **Cellpose** (SAM-based segmentation, preferrably should be run on GPU)

    Output: labeled 3D mask where each nucleus is assigned a unique label.

3. **Quality Control (QC)**
Save .tif mask stack and .png overlays (maximum-intensity projections with labels).
View some masks in the cell output to quickly check performance.

In [ ]:
import numpy as np
from skimage.exposure import rescale_intensity
from skimage.transform import resize
from scipy.ndimage import gaussian_filter

# extract nucleus channel from demo image
nucleus = all_volumes[demo]["channels"]["nucleus"].astype("float32")

# rescale intensity to 0-1
nucleus = rescale_intensity(nucleus, out_range=(0, 1))

# gaussian smoothing
nucleus = gaussian_filter(nucleus, sigma=sigma_vox)

# downsample in XY
z, y, x = nucleus.shape
nucleus = resize(nucleus, (z, int(y * downsize_factor), int(x * downsize_factor)), 
                 anti_aliasing=True).astype("float32")

print("Preprocessed nucleus shape:", nucleus.shape)

# Z resolution is typically worse than XY (due to the PSF)
# we rescale to make voxels isotropic before segmentation

sz, sy, sx = voxel_size_um
min_voxel = min(voxel_size_um)
scale_factors = np.array([sz, sy, sx]) / min_voxel

new_shape = np.round(np.array(nucleus.shape) * scale_factors).astype(int)

nucleus_iso = resize(
    nucleus,
    new_shape,
    anti_aliasing=True,
    preserve_range=True
).astype("float32")

print("Original shape:", nucleus.shape)
print("Isotropic shape:", nucleus_iso.shape)

In [ ]:
# 2. nucleus segmentation


from cellpose import models

model = models.CellposeModel(gpu=gpu)

nuclei_mask, flows, styles = model.eval(
    nucleus_iso,        # the preprocessed nucleus channel isometric
    z_axis=0,       # Z, Y, X order
    do_3D=True      # enables 3D segmentation
)

print("Nuclei found:", nuclei_mask.max())  # max label = number of nuclei

In [ ]:
### visualise mask in the middle region

z_mid = nuclei_mask.shape[0] // 2

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(nucleus_iso[z_mid], cmap="gray")
axes[0].set_title("nucleus (preprocessed)")
axes[1].imshow(nucleus_iso[z_mid], cmap="gray")
axes[1].imshow(np.ma.masked_where(nuclei_mask[z_mid] == 0, nuclei_mask[z_mid]), 
               cmap="autumn", alpha=0.5)
axes[1].set_title("nucleus + mask")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()


b) ** Cytoplasm **

1. Preprocessing: normalise cytoplasm channel, Gaussian filtering typically disabled in "intensity" mode

2. Segmentation

- Cytoplasm segmented via watershed, seeded by nucleus masks.

- Two strategies available:
    - "intensity" (default, based on cytoplasmic intensity)
    - "membrane" (alternative, based on membrane-labeled channels)

3. Saving Results: store cytoplasm mask in results dictionary, save .tif mask and .png overlay for QC

In [ ]:
# ---------------------------------------------------------------------------
# Parameters for the cytoplasm
# ---------------------------------------------------------------------------

cytoplams_sigma_um = 1          # gaussian smoothing in µm

# convert sigma from µm to voxels (accounts for non-isotropic voxel size)
cytoplams_sigma_um = tuple(cytoplams_sigma_um / v for v in voxel_size_um)  # (1.0, 2.08, 2.08)

mode = 'membrane'   # chose 'intensity' if you don't have a membrane marker
# organelle
gaussian_organelle = False      # apply gaussian blue to organelle channel
gaussian_sigma_organelle = [1,1,1]   # Standard deviation for Gaussian smoothing (µm units)

background_subtract = False # true/false: whether to apply background subtraction
normalize = False            # true/false: intensity normalization


In [ ]:
# --- 3.1 cytoplasm channel preprocessing ---
# we already

# extract cytoplasm channel from demo image
cytoplasm = all_volumes[demo]["channels"]["cytoplasm"].astype("float32")

# rescale intensity to 0-1
cytoplasm = rescale_intensity(cytoplasm, out_range=(0, 1))

# gaussian smoothing
cytoplasm = gaussian_filter(cytoplasm, sigma=sigma_vox)

# downsample in XY
z, y, x = cytoplasm.shape
cytoplasm = resize(cytoplasm, (z, int(y * downsize_factor), int(x * downsize_factor)), 
                 anti_aliasing=True).astype("float32")

print("Preprocessed cytoplasm shape:", cytoplasm.shape)

# Z resolution is typically worse than XY (due to the PSF)
# we rescale to make voxels isotropic before segmentation

# sz, sy, sx = voxel_size_um
# min_voxel = min(voxel_size_um)
# scale_factors = np.array([sz, sy, sx]) / min_voxel

new_shape_cytoplasm = np.round(np.array(cytoplasm.shape) * scale_factors).astype(int)

cytoplasm_iso = resize(
    cytoplasm,
    new_shape,
    anti_aliasing=True,
    preserve_range=True
).astype("float32")

print("Original shape:", cytoplasm.shape)
print("Isotropic shape:", cytoplasm_iso.shape)

# ---------------------------------------------------------------------------
# Segmentation of the membrane parameters
# ---------------------------------------------------------------------------


membrane_threshold = 0.2        # Threshold for membrane signal to act as stopping boundary.
min_signal = 0.05       # Minimum normalized intensity for cytoplasm mask (for intensity mode).

# --- 3.2 cytoplasm segmentation (watershed) ---
# Using a cytoplasmic protein or a membrane marker

from skimage.segmentation import watershed
from scipy import ndimage as ndi
from skimage.filters import gaussian

# Segment cytoplasm using nuclei as seeds and cytoplasmic/membrane channel as guidance.
    
if mode == "membrane":
    # binary mask of membrane
    membrane_mask = cytoplasm_iso > membrane_threshold
    distance = ndi.distance_transform_edt(~membrane_mask)
    cytoplasm_labels = watershed(-distance, markers=nuclei_mask, mask=~membrane_mask)

elif mode == "intensity":
    # smooth the channel for better segmentation
    smoothed = gaussian(cytoplasm_iso, sigma=cytoplams_sigma_um) ### pre-processing should be done in the separate step prior to segmentation
    #smoothed = cyto_channel 

    # cytoplasmic regions to include
    cytoplasm_mask = smoothed > min_signal
    inverted = -smoothed
    cytoplasm_labels = watershed(inverted, markers=nuclei_mask, mask=cytoplasm_mask)

else:
    raise ValueError("Invalid mode. Choose 'membrane' or 'intensity'.")

In [ ]:
### Visualize middle section
### does it make sense?

z_mid = cytoplasm_mask.shape[0] // 2

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(cytoplasm_iso[z_mid], cmap="gray")
axes[0].set_title("cytoplasm (preprocessed)")
axes[1].imshow(cytoplasm_iso[z_mid], cmap="gray")
axes[1].imshow(np.ma.masked_where(cytoplasm_mask[z_mid] == 0, cytoplasm_mask[z_mid]), 
               cmap="autumn", alpha=0.5)
axes[1].set_title("cytoplasm + mask")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

---
#### c) organelle / intracellular structure

This sub-section deals with segmenting/quantifying intracellular structures per cell mask, mapping the mask from intracellular channel to the cell_ID label from cytoplasmic mask. The general overview of the process is as follows: 

4.1 **Preprocessing**
- 4.0: obtain the suggested parameters for the preprocessing (from `suggest_normalization_param` helper function)
- Apply recommended normalization parameters (from `suggest_normalization_param`).  
  - Example: scaling intensity to `[0, 17]`.  
  - (optionally) downsample to match cytoplasm resolution -- required for aligning shapes of different channels  
- Apply Gaussian smoothing (e.g. σ = 1).  

4.2 **Segmentation**
- Use **AllenCell wrappers** (e.g. `dot_2d_slice_by_slice_wrapper` for spotty structures), based on the **lookup tables** found here:
  - General pipelines: https://www.allencell.org/segmenter.html#lookup-table 
  - Examples of notebooks with specific functions: https://github.com/AllenCell/aics-segmentation/tree/main/lookup_table_demo
  - Description of all modules available: https://allencell.github.io/aics-segmentation/aicssegmentation.core.html#   
- Returns binary masks (optionally cleaned by removing very small objects).  
- Save `.tif` masks and `.png` overlays for QC.  

4.3 **Quantification**
- Map segmented intracellular structures to corresponding cytoplasm masks.  
- Compute per-object and/or per-cell features:  
  - **count** (number of spots per cell)  
  - **volume** (µm³)  
  - **intensity** (mean intensity per object/cell)  
- Save results into CSV files for downstream analysis.

In [ ]:
# 4.1 do recommended pre-processing of the channel


# --------------------
# Parameters for pre-procesing         DO WE NEED TO DO THIS???!!!
# --------------------
organelle_sigma_um = 1          # gaussian smoothing in µm

# convert sigma from µm to voxels (accounts for non-isotropic voxel size)
organelle_sigma_vox = tuple(organelle_sigma_um / v for v in voxel_size_um)  # (1.0, 2.08, 2.08)

# --------------------

# extract ths structure channel from demo image
structure  = all_volumes[demo]["channels"]["intracellular"].astype("float32")

# # --- preprocessing ---
# # intensity normalization

# # rescale intensity to 0-1
# structure = rescale_intensity(structure, out_range=(0, 1))

# # gaussian smoothing
# structure = gaussian_filter(structure, sigma=sigma_vox)

# downsample in XY
z, y, x = structure.shape
structure = resize(structure, (z, int(y * downsize_factor), int(x * downsize_factor)), 
                 anti_aliasing=True).astype("float32")

print("Preprocessed structure shape:", structure.shape)

# Z resolution is typically worse than XY (due to the PSF)
# we rescale to make voxels isotropic before segmentation

# sz, sy, sx = voxel_size_um
# min_voxel = min(voxel_size_um)
# scale_factors = np.array([sz, sy, sx]) / min_voxel

new_shape_structure = np.round(np.array(structure.shape) * scale_factors).astype(int)

structure_iso = resize(
    structure,
    new_shape_structure,
    anti_aliasing=True,
    preserve_range=True
).astype("float32")

print("Original shape:", structure.shape)
print("Isotropic shape:", structure_iso.shape)



Example below: segmentation workflow for spotty structures
[based on https://github.com/AllenCell/aics-segmentation/blob/main/lookup_table_demo/playground_spotty.ipynb ]:

About selected algorithms and tuned parameters

Intensity normalization: Parameter intensity_scaling_param has two options: two values, say [A, B], or single value, say [K]. For the first case, A and B are non-negative values indicating that the full intensity range of the stack will first be cut-off into [mean - A * std, mean + B * std] and then rescaled to [0, 1]. The smaller the values of A and B are, the higher the contrast will be. For the second case, K>0 indicates min-max Normalization with an absolute intensity upper bound K (i.e., anything above K will be chopped off and reset as the minimum intensity of the stack) and K=0 means min-max Normalization without any intensity bound.

Parameter for fibrillarin: intensity_scaling_param = [0.5, 18]

Parameter for beta catenin: intensity_scaling_param = [4, 27]

Smoothing

3D gaussian smoothing with gaussian_smoothing_sigma = 1 (same for fibrillarin and beta catenin). The larger the value is, the more the image will be smoothed.

In [ ]:
### example: segmentation workflow for spotty structures 
# [based on https://github.com/AllenCell/aics-segmentation/blob/main/lookup_table_demo/playground_spotty.ipynb]

from aicssegmentation.core.pre_processing_utils import intensity_normalization, image_smoothing_gaussian_3d
from aicssegmentation.core.pre_processing_utils import suggest_normalization_param
# --------------------
# ALLEN FUNCTIONS
# --------------------

# first, check what normalisation parameters are suggested for your data
structure = all_volumes[demo]["channels"]["intracellular"].astype("float32")
suggest_normalization_param(structure)


In [ ]:

# --------------------
# SETTINGS for Allen functions!
# adjust based on suggest_normalization_param output above
# --------------------

intensity_scaling_param = [0, 17]   
gaussian_smoothing_sigma = 1        
# --------------------

# AllenCell intensity normalisation
structure = intensity_normalization(structure, scaling_param=intensity_scaling_param) # # AllenCell function, SHOULD BE ISO!!!!!

# AllenCell gaussian smoothing
structure_smooth = image_smoothing_gaussian_3d(structure, sigma=gaussian_smoothing_sigma) # # AllenCell function

print("Preprocessed structure shape:", structure_smooth.shape)

In [ ]:
### Visualize middle section
### does it make sense?

z_mid = structure_smooth.shape[0] // 2

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(structure[z_mid], cmap="gray")
axes[0].set_title("raw intracellular")
axes[1].imshow(structure_smooth[z_mid], cmap="gray")
axes[1].set_title("after normalisation and smoothing")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

apply 2d spot filter
Parameter syntax: [[scale_1, cutoff_1], [scale_2, cutoff_2], ....]

scale_x is set based on the estimated radius of your target spotty shape. For example, if visually the diameter of the spotty objects is usually 3~4 pixels, then you may want to set scale_x as 1 or something near 1 (like 1.25). Multiple scales can be used, if you have objects of very different sizes.
cutoff_x is a threshold applied on the actual filter reponse to get the binary result. Smaller cutoff_x may yielf fatter segmentation, while larger cutoff_x could be less permisive and yield less objects and slimmer segmentation.
Examples:

Parameter for fibrillarin: s2_param = [[1, 0.01]]

Parameter for beta catenin: s2_param = [[1.5, 0.01]]

In [ ]:
from aicssegmentation.core.seg_dot import dot_2d_slice_by_slice_wrapper

# --------------------
# SETTINGS
# adjust scale based on estimated radius of your spots in pixels
# adjust cutoff to control sensitivity (lower = more objects, higher = fewer)
# --------------------
s2_param = [[0.75, 0.01]]  
# --------------------

# apply 2D spot filter slice by slice
# returns a binary mask (True/False), not labelled
structure_mask = dot_2d_slice_by_slice_wrapper(structure_smooth, s2_param)

print("Spots detected:", structure_mask.sum(), "voxels")

# post-process the image: remove small objects if needed??

In [ ]:
### Visualize middle section
### does it make sense?

z_mid = structure_mask.shape[0] // 2

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(structure_iso[z_mid], cmap="gray")
axes[0].set_title("structure (preprocessed)")
axes[1].imshow(structure_iso[z_mid], cmap="gray")
axes[1].imshow(np.ma.masked_where(structure_mask[z_mid] == 0, structure_mask[z_mid]), 
               cmap="autumn", alpha=0.5)
axes[1].set_title("structure + mask")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()